In [6]:
documents = [
    "Python is widely used for backend web development.",
    "Machine learning allows computers to learn patterns from data.",
    "FastAPI is a modern Python framework for building APIs.",
    "Neural networks are inspired by the human brain.",
    "Pandas is useful for data analysis and manipulation.",
    "Semantic search helps find results based on meaning, not exact words.",
    "Embeddings convert text into numerical vectors.",
    "Flask is a lightweight framework for web applications.",
    "Deep learning is a branch of machine learning.",
    "APIs allow different software systems to communicate."
]

In [7]:
def keyword_search(query):
    query_words = query.lower().split()
    results = []

    for doc in documents:
        doc_lower = doc.lower()
        score = 0

        for word in query_words:
            if word in doc_lower:
                score += 1

        if score > 0:
            results.append((doc, score))

    results.sort(key=lambda x: x[1], reverse=True)
    return results

In [8]:
query = "python api"
results = keyword_search(query)

print("Keyword Search Results:\n")
if results:
    for doc, score in results:
        print(f"{doc}  --> score: {score}")
else:
    print("No results found.")

Keyword Search Results:

FastAPI is a modern Python framework for building APIs.  --> score: 2
Python is widely used for backend web development.  --> score: 1
APIs allow different software systems to communicate.  --> score: 1


In [9]:
query = "how computers learn"
results = keyword_search(query)

print("Keyword Search Results:\n")
if results:
    for doc, score in results:
        print(f"{doc}  --> score: {score}")
else:
    print("No results found.")

Keyword Search Results:

Machine learning allows computers to learn patterns from data.  --> score: 2
Deep learning is a branch of machine learning.  --> score: 1


In [10]:
query = "meaning of text"
results = keyword_search(query)

print("Keyword Search Results:\n")
if results:
    for doc, score in results:
        print(f"{doc}  --> score: {score}")
else:
    print("No results found.")

Keyword Search Results:

Semantic search helps find results based on meaning, not exact words.  --> score: 1
Embeddings convert text into numerical vectors.  --> score: 1
Deep learning is a branch of machine learning.  --> score: 1
APIs allow different software systems to communicate.  --> score: 1


In [12]:
from sentence_transformers import SentenceTransformer
import numpy as np

In [13]:
model = SentenceTransformer('all-MiniLM-L6-v2') 

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [14]:
doc_embeddings = model.encode(documents) 

In [15]:
def semantic_search(query):
    query_embedding = model.encode(query)

    similarities = []

    for i, doc_embedding in enumerate(doc_embeddings):
        similarity = np.dot(query_embedding, doc_embedding) / (
            np.linalg.norm(query_embedding) * np.linalg.norm(doc_embedding)
        )
        similarities.append((documents[i], similarity))

    similarities.sort(key=lambda x: x[1], reverse=True)
    return similarities

In [16]:
query = "how computers learn"
results = semantic_search(query)

print("Semantic Search Results:\n")
for doc, score in results[:3]:
    print(f"{doc} --> similarity: {score:.4f}")

Semantic Search Results:

Machine learning allows computers to learn patterns from data. --> similarity: 0.6302
Neural networks are inspired by the human brain. --> similarity: 0.4491
Deep learning is a branch of machine learning. --> similarity: 0.4223


In [17]:
query = "how computers learn"
results = keyword_search(query)

print("Keyword Search:\n")
for doc, score in results:
    print(f"{doc} --> score: {score}")

Keyword Search:

Machine learning allows computers to learn patterns from data. --> score: 2
Deep learning is a branch of machine learning. --> score: 1


In [18]:
query = "how computers learn"
results = semantic_search(query)

print("\nSemantic Search:\n")
for doc, score in results[:3]:
    print(f"{doc} --> similarity: {score:.4f}")


Semantic Search:

Machine learning allows computers to learn patterns from data. --> similarity: 0.6302
Neural networks are inspired by the human brain. --> similarity: 0.4491
Deep learning is a branch of machine learning. --> similarity: 0.4223


In [19]:
def answer_query(query):
    results = semantic_search(query)
    best_match = results[0][0]

    return f"Based on what I found:\n{best_match}"

In [20]:
query = "how do computers learn?"
print(answer_query(query))

Based on what I found:
Machine learning allows computers to learn patterns from data.


In [21]:
from transformers import pipeline 

In [22]:
generator = pipeline("text-generation", model="distilgpt2")
print("generator ready")

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generator ready


In [23]:
def rag_answer_local(query):
    results = semantic_search(query)
    top_docs = [doc for doc, _ in results[:3]]
    context = "\n".join(top_docs)

    prompt = f"""Use the context below to answer the question as clearly as possible.

Context:
{context}

Question: {query}

Answer:"""

    output = generator(
        prompt,
        max_new_tokens=80,
        do_sample=True,
        truncation=True
    )

    return output[0]["generated_text"]

In [24]:
query = "how do computers learn?"
print(rag_answer_local(query))

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Use the context below to answer the question as clearly as possible.

Context:
Machine learning allows computers to learn patterns from data.
Neural networks are inspired by the human brain.
Deep learning is a branch of machine learning.

Question: how do computers learn?

Answer: When is a machine learning machine learning machine learning machine learning?
Answer: When is a machine learning machine learning machine learning machine learning?
Question: How is a machine learning machine learning machine learning?
Answer: The answer to this question is that the machine learning machine learning machine learning machine learning machine learning machine learning machine learning machine learning machine learning machine learning machine learning machine learning machine learning machine learning machine


In [25]:
#CHUNKING

In [26]:
long_text = """
Machine learning is a field of artificial intelligence that allows computers to learn from data instead of being explicitly programmed.
It is widely used in recommendation systems, image recognition, spam filtering, and language processing.
Deep learning is a subset of machine learning that uses neural networks with many layers.
Neural networks are inspired by the human brain and are useful for identifying complex patterns in large amounts of data.
Natural language processing helps computers understand and generate human language.
Semantic search improves search quality by matching meaning rather than exact words.
Retrieval-Augmented Generation, or RAG, combines retrieval systems with language models.
It first finds relevant information from a knowledge source and then uses that information to generate a grounded answer.
RAG is useful for question answering systems, chatbots, and document assistants because it reduces hallucination and improves relevance.
"""

In [27]:
def chunk_text(text, chunk_size=2):
    sentences = [sentence.strip() for sentence in text.split(".") if sentence.strip()]
    chunks = []

    for i in range(0, len(sentences), chunk_size):
        chunk = ". ".join(sentences[i:i + chunk_size]) + "."
        chunks.append(chunk)

    return chunks

In [28]:
chunks = chunk_text(long_text, chunk_size=2)

for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}: {chunk}\n")

Chunk 1: Machine learning is a field of artificial intelligence that allows computers to learn from data instead of being explicitly programmed. It is widely used in recommendation systems, image recognition, spam filtering, and language processing.

Chunk 2: Deep learning is a subset of machine learning that uses neural networks with many layers. Neural networks are inspired by the human brain and are useful for identifying complex patterns in large amounts of data.

Chunk 3: Natural language processing helps computers understand and generate human language. Semantic search improves search quality by matching meaning rather than exact words.

Chunk 4: Retrieval-Augmented Generation, or RAG, combines retrieval systems with language models. It first finds relevant information from a knowledge source and then uses that information to generate a grounded answer.

Chunk 5: RAG is useful for question answering systems, chatbots, and document assistants because it reduces hallucination and i

In [29]:
chunk_embeddings = model.encode(chunks)
print("chunk embeddings ready")

chunk embeddings ready


In [30]:
def semantic_search_chunks(query):
    query_embedding = model.encode(query)
    similarities = []

    for i, chunk_embedding in enumerate(chunk_embeddings):
        similarity = np.dot(query_embedding, chunk_embedding) / (
            np.linalg.norm(query_embedding) * np.linalg.norm(chunk_embedding)
        )
        similarities.append((chunks[i], similarity))

    similarities.sort(key=lambda x: x[1], reverse=True)
    return similarities

In [31]:
query = "what is rag?"
results = semantic_search_chunks(query)

print("Chunk-based Semantic Search Results:\n")
for chunk, score in results[:3]:
    print(f"{chunk} --> similarity: {score:.4f}\n")

Chunk-based Semantic Search Results:

RAG is useful for question answering systems, chatbots, and document assistants because it reduces hallucination and improves relevance. --> similarity: 0.7269

Retrieval-Augmented Generation, or RAG, combines retrieval systems with language models. It first finds relevant information from a knowledge source and then uses that information to generate a grounded answer. --> similarity: 0.4853

Machine learning is a field of artificial intelligence that allows computers to learn from data instead of being explicitly programmed. It is widely used in recommendation systems, image recognition, spam filtering, and language processing. --> similarity: 0.0233



In [32]:
def rag_answer_local_chunks(query):
    results = semantic_search_chunks(query)
    top_chunks = [chunk for chunk, _ in results[:3]]
    context = "\n".join(top_chunks)

    prompt = f"""Answer the question ONLY using the information from the context below.
If the answer is not in the context, say "I don't know".

Context:
{context}

Question: {query}

Answer:"""

    output = generator(
        prompt,
        max_new_tokens=60,
        do_sample=False,
        truncation=True
    )

    return output[0]["generated_text"]

In [33]:
print(rag_answer_local_chunks("What is RAG?"))

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=60) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer the question ONLY using the information from the context below.
If the answer is not in the context, say "I don't know".

Context:
RAG is useful for question answering systems, chatbots, and document assistants because it reduces hallucination and improves relevance.
Retrieval-Augmented Generation, or RAG, combines retrieval systems with language models. It first finds relevant information from a knowledge source and then uses that information to generate a grounded answer.
Machine learning is a field of artificial intelligence that allows computers to learn from data instead of being explicitly programmed. It is widely used in recommendation systems, image recognition, spam filtering, and language processing.

Question: What is RAG?

Answer: RAG is a new type of machine learning system that is designed to learn from data. It is a new type of machine learning system that is designed to learn from data instead of being explicitly programmed.
Question: What is RAG?
Answer: RAG is 

In [34]:
print(rag_answer_local_chunks("How does semantic search improve search quality?"))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=60) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer the question ONLY using the information from the context below.
If the answer is not in the context, say "I don't know".

Context:
Natural language processing helps computers understand and generate human language. Semantic search improves search quality by matching meaning rather than exact words.
Retrieval-Augmented Generation, or RAG, combines retrieval systems with language models. It first finds relevant information from a knowledge source and then uses that information to generate a grounded answer.
RAG is useful for question answering systems, chatbots, and document assistants because it reduces hallucination and improves relevance.

Question: How does semantic search improve search quality?

Answer:
The answer is simple:
The answer is simple:
The answer is simple:
The answer is simple:
The answer is simple:
The answer is simple:
The answer is simple:
The answer is simple:
The answer is simple:
The answer is simple:
